In [ ]:
# Restart Python kernel to ensure clean state
# dbutils.library.restartPython()

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import sha2, concat_ws, xxhash64

In [ ]:
import os
import sys
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir,  "..","..",".."))
sys.path.append(project_root)
from src.carburants.fuel_price_utils import download_csv_file, read_csv_file

In [ ]:
url = "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/prix-des-carburants-en-france-flux-instantane-v2/exports/csv?use_labels=true"
path = "/Volumes/fuel_price_dev/landing/data/carburants"

In [ ]:
# Download the latest data file (commented out - using existing file)
file = download_csv_file(url, path)

## Load source data

Read the fuel prices CSV file from the Unity Catalog volume.

In [ ]:
# Path to the fuel prices CSV file in Unity Catalog volume
#file = "/Volumes/fuel_price_dev/landing/data/carburants/prix-des-carburants-en-france-flux-instantane-v2.csv"

schema_df = StructType([
    StructField("id", IntegerType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("Code postal", IntegerType(), True),
    StructField("pop", StringType(), True),
    StructField("Adresse", StringType(), True),
    StructField("Ville", StringType(), True),
    StructField("horaires", StringType(), True),
    StructField("services", StringType(), True),
    StructField("prix", StringType(), True),
    StructField("rupture", StringType(), True),
    StructField("geom", StringType(), True),
    StructField("Prix Gazole mis à jour le", TimestampType(), True),
    StructField("Prix Gazole", DoubleType(), True),
    StructField("Prix SP95 mis à jour le", TimestampType(), True),
    StructField("Prix SP95", DoubleType(), True),
    StructField("Prix E85 mis à jour le", TimestampType(), True),
    StructField("Prix E85", DoubleType(), True),
    StructField("Prix GPLc mis à jour le", TimestampType(), True),
    StructField("Prix GPLc", DoubleType(), True),
    StructField("Prix E10 mis à jour le", TimestampType(), True),
    StructField("Prix E10", DoubleType(), True),
    StructField("Prix SP98 mis à jour le", TimestampType(), True),
    StructField("Prix SP98", DoubleType(), True),
    StructField("Début rupture e10 (si temporaire)", TimestampType(), True),
    StructField("Type rupture e10", StringType(), True),
    StructField("Début rupture sp98 (si temporaire)", TimestampType(), True),
    StructField("Type rupture sp98", StringType(), True),
    StructField("Début rupture sp95 (si temporaire)", TimestampType(), True),
    StructField("Type rupture sp95", StringType(), True),
    StructField("Début rupture e85 (si temporaire)", TimestampType(), True),
    StructField("Type rupture e85", StringType(), True),
    StructField("Début rupture GPLc (si temporaire)", TimestampType(), True),
    StructField("Type rupture GPLc", StringType(), True),
    StructField("Début rupture gazole (si temporaire)", TimestampType(), True),
    StructField("Type rupture gazole", StringType(), True),
    StructField("Carburants disponibles", StringType(), True),
    StructField("Carburants indisponibles", StringType(), True),
    StructField("Carburants en rupture temporaire", StringType(), True),
    StructField("Carburants en rupture definitive", StringType(), True),
    StructField("Automate 24-24 (oui/non)", StringType(), True),
    StructField("Services proposés", StringType(), True),
    StructField("Département", StringType(), True),
    StructField("code_departement", StringType(), True),
    StructField("Région", StringType(), True),
    StructField("code_region", IntegerType(), True),
    StructField("horaires détaillés", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])
# Read CSV file into a Spark DataFrame
carburants_df = read_csv_file(file, spark, schema_df)

## Define data schemas

Define PySpark schemas for parsing nested JSON columns (services, prix, rupture).

In [ ]:

schema_prix = ArrayType(StructType([
    StructField("@nom", StringType(), True),      
    StructField("@id", StringType(), True),       
    StructField("@maj", StringType(), True),     
    StructField("@valeur", StringType(), True),   
]))


schema_rupture = ArrayType(StructType([
    StructField("@nom", StringType(), True),     
    StructField("@id", StringType(), True),     
    StructField("@debut", StringType(), True),      
    StructField("@fin", StringType(), True),        
    StructField("@type", StringType(), True),   
]))

## Transformation functions

Define functions to build dimension and fact tables from the raw data.

In [ ]:

# ============================================================================
# DIMENSION TABLE BUILDERS
# ============================================================================

# define full extracting metadata
dbutils.widgets.text("date_ingestion", "")
date_ingestion = dbutils.widgets.get("date_ingestion")


def build_dim_geo(df):
    dim_geo_df = df\
        .select(
            "code_region",
            col("Région").alias("region"),
            "code_departement",
            col("Département").alias("departement")
        )\
        .dropDuplicates(["code_departement"])\
        .withColumn("date_ingestion", lit(date_ingestion))\
        .sort("code_departement", ascending=True)
    return dim_geo_df

def build_dim_station(df):
    return df\
    .dropDuplicates(["id"])\
    .withColumn("date_ingestion", lit(date_ingestion))\
        .select(
        col("id").alias("station_id"),
        "adresse",
        "ville",
        col("Code postal").alias("code_postal"),
        col("Automate 24-24 (oui/non)").alias("automate_24h_24"),
        "code_departement",
        "latitude",
        "longitude",
        col("Services proposés").alias("service"),
        "code_region",
        "date_ingestion"
    ).sort("station_id",ascending=True)

def build_dim_carburant(df):
    dim_carburant_df = df\
        .withColumn("prix_parsed", from_json(col("prix"), schema_prix))\
        .withColumn("carburant", explode(col("prix_parsed")))\
        .select(
            col("carburant.@id").alias("id"),
            col("carburant.@nom").alias("nom")
        )\
        .dropDuplicates(["id"])\
        .withColumn("date_ingestion", lit(date_ingestion))\
        .sort("id", ascending=True)
    return dim_carburant_df


In [ ]:
# ============================================================================
# FACT TABLE 
# ============================================================================

def build_fait_prix(df):

    df = df.withColumn("prix_parsed", from_json(col("prix"), schema_prix))\
           .withColumn("carburant", explode(col("prix_parsed")))\
           .withColumn("date_ingestion", lit(date_ingestion))\
           .withColumn("id_fct_pr", sha2(concat_ws("||", "id", "carburant.@id"), 256))
    return df\
     .select(
        col("id_fct_pr"),
        col("id").alias("station_id"),
        col("carburant.@id").alias("carburant_id"),
        col("carburant.@nom").alias("nom_carburant"),
        col("carburant.@maj").alias("date_maj"),
        col("carburant.@valeur").alias("prix"),
        "date_ingestion"
    )


def build_fait_rupture(df):
   
    rupture_parsed_df = df\
                        .withColumn("rupture_parsed",from_json(col("rupture"), schema_rupture))
    rupture_exploded_df = rupture_parsed_df\
                        .withColumn("rupture_exploded", explode(col('rupture_parsed')))\
                        .withColumn("date_ingestion", lit(date_ingestion))\
                        .withColumn("id_fct_rpt", sha2(concat_ws("||", "id", "rupture_exploded.@id"), 256))
    return rupture_exploded_df\
        .select(
            "id_fct_rpt",
            col("id").alias("station_id"),
            col("rupture_exploded.@id").alias("id_carburant"),
             col("rupture_exploded.@nom").alias("nom_carburant"),
            col("rupture_exploded.@debut").alias("debut_rupture"),
            col("rupture_exploded.@fin").alias("fin_rupture"),
            col("rupture_exploded.@type").alias("type_rupture"),
            "date_ingestion")


In [ ]:
# ============================================================================
# BUILD DIMENSIONS 
# ============================================================================
dim_geo     = build_dim_geo(carburants_df)
dim_station = build_dim_station(carburants_df)
dim_carburant = build_dim_carburant(carburants_df)
# ============================================================================
# BUILD FACT TABLES
# ============================================================================
fait_prix    = build_fait_prix(carburants_df).drop('nom_carburant')
fait_rupture = build_fait_rupture(carburants_df).drop('nom_carburant')

## Build and persist dimension tables


In [ ]:
dim_geo.write\
        .format("delta")\
        .option("overwriteSchema", "true")\
        .mode("overwrite")\
        .saveAsTable("fuel_price_dev.bronze.brze_dim_geo")

dim_station.write\
        .format("delta")\
        .option("overwriteSchema", "true")\
        .mode("overwrite")\
        .saveAsTable("fuel_price_dev.bronze.brze_dim_station")

dim_carburant.write\
        .format("delta")\
        .option("overwriteSchema", "true")\
        .mode("overwrite")\
        .saveAsTable("fuel_price_dev.bronze.brze_dim_carburant")

## Build and persist fact tables

In [ ]:
fait_prix.write\
        .format("delta")\
        .option("overwriteSchema", "true")\
        .mode("overwrite")\
        .saveAsTable("fuel_price_dev.bronze.brze_fait_prix")

fait_rupture.write\
        .format("delta")\
        .option("overwriteSchema", "true")\
        .mode("overwrite")\
        .saveAsTable("fuel_price_dev.bronze.brze_fait_rupture")